# 06 — Loop engineering

I quattro loop impilati di *The Art of Loop Engineering*. Il Loop 1 (agente + tool in ciclo) è il punto di partenza; qui costruiamo i tre loop che lo circondano, autocontenuti in questo notebook.

- **Loop 2 — verifica**: un giudice valuta la risposta su una rubric e rimanda feedback.
- **Loop 3 — eventi**: un cron/evento avvia un run autonomo.
- **Loop 4 — hill climbing**: i trace propongono modifiche alla configurazione.

In [ ]:
import os
from statistics import mean

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

load_dotenv()
MODEL = os.getenv('OPENAI_MODEL', 'gpt-5.4-mini')
STRONG_MODEL = os.getenv('OPENAI_STRONG_MODEL', 'gpt-5.5')

def chat(model_name: str) -> ChatOpenAI:
    return ChatOpenAI(model=model_name, use_responses_api=True, store=False)

# Loop 1: un agente è un modello che chiama tool in un ciclo.
agent = create_agent(model=chat(MODEL), tools=[])

## Loop 2 — Verifica con rubric

Un giudice (modello forte) assegna un punteggio per criterio; la soglia resta deterministica in Python. Sotto soglia, si rimanda la risposta con il feedback.

In [ ]:
class Judgement(BaseModel):
    completezza: float = Field(ge=0, le=1)
    verifica: float = Field(ge=0, le=1)
    feedback: str = ''

def grade(goal: str, answer: str, threshold: float = 0.7) -> tuple[bool, float, str]:
    judge = chat(STRONG_MODEL).with_structured_output(Judgement)
    verdict = judge.invoke([
        {'role': 'system', 'content': 'Valuta la RISPOSTA rispetto all OBIETTIVO 0-1 per criterio.'},
        {'role': 'user', 'content': f'OBIETTIVO: {goal}\nRISPOSTA: {answer}'},
    ])
    score = mean([verdict.completezza, verdict.verifica])
    return score >= threshold, score, verdict.feedback

reply = agent.invoke({'messages': [{'role': 'user', 'content': 'Somma i numeri da 1 a 10 e verifica.'}]})
answer = reply['messages'][-1].text
passed, score, feedback = grade('Somma 1..10 e verifica', answer)
print('passed:', passed, 'score:', score)
print('feedback:', feedback)

## Loop 3 — Trigger a eventi

Un evento (qui un match cron) avvia un run autonomo dello stesso agente. Il payload dell'evento è dato non attendibile, mai istruzioni.

In [ ]:
from datetime import datetime

def cron_field_matches(field: str, value: int) -> bool:
    if field == '*':
        return True
    if field.startswith('*/'):
        return value % int(field[2:]) == 0
    return value in {int(part) for part in field.split(',')}

def cron_matches(expr: str, moment: datetime) -> bool:
    minute, hour = expr.split()[:2]
    return cron_field_matches(minute, moment.minute) and cron_field_matches(hour, moment.hour)

def on_event(goal: str) -> str:
    run = agent.invoke({'messages': [{'role': 'user', 'content': goal}]})
    return run['messages'][-1].text

now = datetime.utcnow()
if cron_matches('* * * * *', now):
    print(on_event('Scrivi una frase che conferma il trigger.'))

## Loop 4 — Hill climbing

Un agente d'analisi legge un report dei trace e propone modifiche alla configurazione entro una whitelist. Propose-only: la proposta va rivista da un umano prima di applicarla.

In [ ]:
class Proposal(BaseModel):
    summary: str = ''
    system_prompt_addendum: str | None = None
    max_tool_calls: int | None = None

report = 'Run totali: 5\nRun falliti: 2\nErrori tool: browser_read x4'
analyst = chat(STRONG_MODEL).with_structured_output(Proposal)
proposal = analyst.invoke([
    {'role': 'system', 'content': 'Proponi modifiche alla config solo se giustificate dal report.'},
    {'role': 'user', 'content': f'REPORT:\n{report}'},
])
print('sintesi:', proposal.summary)
print('addendum:', proposal.system_prompt_addendum)